# P05 · Confound controls for the encoder comparisons (development events)

The readiness review found two confounds in the encoder comparisons of p04, both cheap to remove:

1. **Precision.** The DINOv3 ViT-L models ran in fp32 (fp16 overflows) and the ViT-B in fp16. Here DINOv3 ViT-B/16
   is rerun in **fp32**, so ViT-L vs ViT-B is compared at the same precision, and fp32 vs fp16 for ViT-B shows how
   much precision alone changes.
2. **Normalization.** The satellite ViT-L was normalized with its own published statistics, the web ViT-L with
   ImageNet's. Here the **web ViT-L is rerun with the satellite statistics**, so satellite vs web is compared with
   identical input normalization.

Everything else is as in p04: frozen encoders, outline pooling, native resolution, the same leave-one-event-out
probes. Exploratory; development and secondary events only.

* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN` (licence for the
  DINOv3 models already accepted for p04). Needs p01, p02 (DINOv3 ViT-B features) and p04 (ViT-L features).
* **Time:** about 60–70 minutes (ViT-B fp32 ~15 min, web ViT-L ~35 min, bench run ~10 min); about 3 compute units.
* **Outputs:** features in `02_models/features/<date>_confound_controls/`, the run in this run's folder.

In [ ]:
NOTEBOOK_NAME = "p05_confound_controls"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU, Hugging Face token, inputs from p02 and p04
import glob, os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
from huggingface_hub import model_info
for mid in ["facebook/dinov3-vitl16-pretrain-lvd1689m", "facebook/dinov3-vitl16-pretrain-sat493m"]:
    try:
        model_info(mid, token=os.environ["HF_TOKEN"]); print("access OK:", mid)
    except Exception as ex:
        raise RuntimeError(f"No access to {mid}: open https://huggingface.co/{mid} with the HF_TOKEN account, "
                           "accept the licence, then Run all again.") from ex
P02_FEATS = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
P04_DIR = sorted(glob.glob(f"{PROJECT}/02_models/features/*_more_encoders"))
assert os.path.exists(P02_FEATS), f"missing {P02_FEATS}: run p02 first"
assert P04_DIR, "missing the p04 features: run p04 first"
P04_L, P04_SAT = f"{P04_DIR[-1]}/dinov3_vitl16.parquet", f"{P04_DIR[-1]}/dinov3_vitl16_sat.parquet"
assert os.path.exists(P04_L) and os.path.exists(P04_SAT), "p04 ViT-L features missing"
print("GPU:", torch.cuda.get_device_name(0), "| p04 features:", P04_DIR[-1])

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import datetime, gc
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_confound_controls"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# Features, outline pooling, native resolution: DINOv3 ViT-B/16 in fp32; web DINOv3 ViT-L/16 with satellite statistics
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile, write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
FEATS, timing, revisions = {}, [], {}
for name in ["dinov3_vitb16_fp32", "dinov3_vitl16_satnorm"]:
    enc = get_encoder(name)
    revisions[name] = enc.weights_revision
    print(name, "| dtype:", enc.dtype, "| weights:", enc.weights_revision)
    f, runs = extract_at_resolutions(use, lambda t: read_bright_tile(ROOT, t), enc, ["outline"], [None],
                                     f"/content/features/{name}", events, log=print)
    FEATS[name] = str(write_features(f, f"{FEAT_DIR}/{name}.parquet"))
    timing.append(runs_table(runs))
    del f, enc; gc.collect(); torch.cuda.empty_cache()
timing = pd.concat(timing, ignore_index=True)
timing.to_csv(f"{RUN}/timing.csv", index=False)
pd.Series(revisions, name="weights_revision").to_csv(f"{RUN}/weights_revisions.csv")
display(timing); print(revisions)

In [ ]:
# Bench run: the four confound-controlled comparisons
del use; gc.collect()
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
B16, B32 = "dinov3_vitb16@native", "dinov3_vitb16_fp32@native"
L16, LSN, SAT = "dinov3_vitl16@native", "dinov3_vitl16_satnorm@native", "dinov3_vitl16_sat@native"
comps = [
    ComparisonSpec((B32, "outline"), (B16, "outline"), "DINOv3 ViT-B: fp32 vs fp16"),
    ComparisonSpec((LSN, "outline"), (L16, "outline"), "DINOv3 ViT-L web: satellite vs ImageNet statistics"),
    ComparisonSpec((SAT, "outline"), (LSN, "outline"), "DINOv3 ViT-L: satellite vs web, same statistics"),
    ComparisonSpec((L16, "outline"), (B32, "outline"), "DINOv3 web, fp32: ViT-L vs ViT-B"),
]
out = run_benchmark([P02_FEATS, P04_L, P04_SAT, FEATS["dinov3_vitb16_fp32"], FEATS["dinov3_vitl16_satnorm"]],
                    comps, events, f"{RUN}/bench_run", n_boot=2000, only_compared=True)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))
m = pd.read_csv(f"{out}/event_metrics.csv")
pv = m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc"); pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string())
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "estimate", "lower", "upper", "verdict", "sign_flip_p"]].round(4).to_string(index=False))

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))